These five datasets are already scaled and pivoted. However, some scales need to be normalized to a 1-5 scale.

In [2]:
import pandas as pd

In [3]:
df_skills = pd.read_csv('transformed data/1-4-Skills Pivoted.csv')
df_abilities = pd.read_csv('transformed data/1-5-Abilities Pivoted.csv')
df_activites = pd.read_csv('transformed data/1-7-Activities Pivoted.csv')
df_knowledge = pd.read_csv('transformed data/1-8-Knowledge Pivoted.csv')
df_work_context = pd.read_csv('transformed data/1-9-Work Context Frequency Weighted Average.csv')

In [4]:
dfs = [df_skills, df_abilities, df_activites, df_knowledge, df_work_context]

'Level' columns are on various scales: 1-7 and others. All will be normalized to 1-5

'Importance' columns are already 1-5.

The work context dataframe is already 1-5, so it will be left alone.

In [5]:
import pandas as pd
import numpy as np
import re

def rescale_onet_levels_to_1_5(
    df: pd.DataFrame,
    level_regex: str = r"(?:^|[\s_\-(:])level(?:$|[\s_\-)\]:])",   # matches "level" as a word-ish token
    importance_regex: str = r"(?:^|[\s_\-(:])importance(?:$|[\s_\-)\]:])",
    id_cols: list[str] | None = None,
    clip: bool = True
) -> pd.DataFrame:
    """
    Convert O*NET Level columns (0–7) to 1–5 using: new = 1 + (x/7)*4
    Leaves Importance columns (1–5) unchanged.
    
    level_regex / importance_regex are matched against column names (case-insensitive).
    id_cols are columns to never touch (e.g., SOC code, Title).
    """
    if id_cols is None:
        id_cols = ["O*NET-SOC Code", "Title"]

    out = df.copy()

    # numeric columns only (avoid trying to scale text)
    numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()

    level_pat = re.compile(level_regex, flags=re.IGNORECASE)
    imp_pat   = re.compile(importance_regex, flags=re.IGNORECASE)

    # Identify Level columns by name (among numeric cols), excluding ID cols
    level_cols = [c for c in numeric_cols if c not in id_cols and level_pat.search(c)]

    # (Optional) sanity: if you want to avoid touching columns that are clearly importance
    level_cols = [c for c in level_cols if not imp_pat.search(c)]

    # Transform Level 0–7 -> 1–5
    out[level_cols] = 1 + (out[level_cols] / 7.0) * 4.0

    if clip:
        # keeps tiny rounding drift from producing 0.9999 or 5.0001
        out[level_cols] = out[level_cols].clip(lower=1, upper=5)

    return out, level_cols

In [6]:
df_skills_scaled, skills_level_cols = rescale_onet_levels_to_1_5(df_skills)
df_abilities_scaled, abilities_level_cols = rescale_onet_levels_to_1_5(df_abilities)
df_activities_scaled, activities_level_cols = rescale_onet_levels_to_1_5(df_activites)
df_knowledge_scaled, knowledge_level_cols = rescale_onet_levels_to_1_5(df_knowledge)

# Work context is already 1–5, so typically no changes:
df_work_context_scaled = df_work_context.copy()

# Check to see the new dataframes look right

In [7]:
df_skills_scaled

,O*NET-SOC Code,Title,Active Learning Importance,Active Learning Level,Active Listening Importance,Active Listening Level,Complex Problem Solving Importance,Complex Problem Solving Level,Coordination Importance,Coordination Level,...,Systems Evaluation Importance,Systems Evaluation Level,Technology Design Importance,Technology Design Level,Time Management Importance,Time Management Level,Troubleshooting Importance,Troubleshooting Level,Writing Importance,Writing Level
0,11-1011.00,Chief Executives,3.75,3.571429,4.00,3.714286,4.38,3.788571,4.25,3.788571,...,4.25,3.857143,1.75,1.502857,4.00,3.714286,1.50,1.285714,4.12,3.502857
1,11-1011.03,Chief Sustainability Officers,3.75,3.217143,4.00,3.285714,4.00,3.354286,3.75,3.217143,...,3.88,3.285714,1.88,1.640000,3.38,3.217143,1.00,1.000000,4.12,3.428571
2,11-1021.00,General and Operations Managers,3.62,3.142857,4.00,3.354286,3.62,3.217143,3.88,3.285714,...,3.12,2.857143,1.50,1.354286,3.62,3.217143,1.75,1.571429,3.50,3.217143
3,11-2011.00,Advertising and Promotions Managers,3.25,3.354286,4.12,3.354286,3.50,3.217143,3.50,3.354286,...,3.12,3.142857,1.75,1.428571,3.50,3.217143,1.00,1.000000,3.75,3.217143
4,11-2021.00,Marketing Managers,3.88,3.354286,3.88,3.354286,3.62,3.217143,3.50,3.142857,...,3.50,3.142857,1.75,1.502857,3.50,3.142857,1.00,1.000000,3.25,3.217143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.88,2.360000,3.12,2.714286,3.00,2.714286,3.00,2.645714,...,2.00,2.211429,1.88,1.640000,3.00,2.571429,3.12,2.782857,3.00,2.645714
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",2.88,2.645714,3.12,2.714286,2.88,2.645714,2.88,2.714286,...,2.12,2.211429,1.88,1.502857,3.00,2.645714,3.00,2.645714,2.88,2.571429
891,53-7073.00,Wellhead Pumpers,2.38,2.211429,2.88,2.571429,3.00,2.645714,2.25,2.360000,...,2.00,1.788571,1.50,1.285714,2.75,2.571429,3.12,2.714286,2.62,2.211429
892,53-7081.00,Refuse and Recyclable Material Collectors,2.25,1.925714,2.88,2.428571,2.38,2.211429,2.62,2.428571,...,1.38,1.428571,1.00,1.000000,2.50,2.360000,2.50,2.142857,2.50,2.142857


In [8]:
df_abilities_scaled

,O*NET-SOC Code,Title,Arm-Hand Steadiness Importance,Arm-Hand Steadiness Level,Auditory Attention Importance,Auditory Attention Level,Category Flexibility Importance,Category Flexibility Level,Control Precision Importance,Control Precision Level,...,Visual Color Discrimination Importance,Visual Color Discrimination Level,Visualization Importance,Visualization Level,Wrist-Finger Speed Importance,Wrist-Finger Speed Level,Written Comprehension Importance,Written Comprehension Level,Written Expression Importance,Written Expression Level
0,11-1011.00,Chief Executives,1.38,1.285714,2.12,2.142857,3.50,3.285714,1.75,1.428571,...,2.00,2.142857,3.00,2.931429,1.38,1.285714,4.25,3.788571,4.12,3.714286
1,11-1011.03,Chief Sustainability Officers,1.00,1.000000,2.00,2.000000,3.12,3.000000,1.50,1.285714,...,2.12,2.142857,2.75,2.645714,1.12,1.068571,4.00,3.428571,4.12,3.502857
2,11-1021.00,General and Operations Managers,1.62,1.502857,2.12,2.142857,3.38,2.857143,1.12,1.068571,...,1.88,2.000000,2.75,2.571429,1.75,1.502857,4.00,3.285714,4.00,3.285714
3,11-2011.00,Advertising and Promotions Managers,1.38,1.285714,1.75,1.714286,3.38,3.217143,1.12,1.068571,...,2.88,2.571429,3.25,2.782857,1.75,1.428571,4.00,3.285714,3.88,3.285714
4,11-2021.00,Marketing Managers,1.12,1.068571,1.88,1.925714,3.25,3.068571,1.00,1.000000,...,2.88,2.645714,3.00,2.714286,1.62,1.428571,4.00,3.354286,3.88,3.354286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,3.88,2.782857,3.00,2.714286,3.00,2.714286,3.50,2.782857,...,2.88,2.360000,2.75,2.571429,2.00,2.142857,3.12,2.645714,3.00,2.645714
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",3.12,2.782857,3.00,3.000000,3.00,2.782857,3.38,2.857143,...,3.00,2.857143,3.00,2.857143,2.62,2.211429,3.12,2.714286,3.25,2.645714
891,53-7073.00,Wellhead Pumpers,3.00,2.645714,2.25,2.497143,2.88,2.714286,3.25,3.285714,...,2.00,1.925714,2.75,2.782857,2.12,1.925714,2.88,2.571429,2.62,2.360000
892,53-7081.00,Refuse and Recyclable Material Collectors,3.25,2.285714,2.75,2.285714,2.75,2.360000,2.88,2.645714,...,2.88,2.571429,2.50,2.285714,1.62,1.571429,2.50,2.285714,2.50,2.142857


In [9]:
df_activities_scaled

,O*NET-SOC Code,Title,Analyzing Data or Information Importance,Analyzing Data or Information Level,Assisting and Caring for Others Importance,Assisting and Caring for Others Level,Coaching and Developing Others Importance,Coaching and Developing Others Level,Communicating with People Outside the Organization Importance,Communicating with People Outside the Organization Level,...,Staffing Organizational Units Importance,Staffing Organizational Units Level,Thinking Creatively Importance,Thinking Creatively Level,Training and Teaching Others Importance,Training and Teaching Others Level,Updating and Using Relevant Knowledge Importance,Updating and Using Relevant Knowledge Level,Working with Computers Importance,Working with Computers Level
0,11-1011.00,Chief Executives,4.38,3.765714,3.31,2.771429,4.71,4.342857,4.55,4.125714,...,3.78,3.742857,4.32,3.542857,4.01,3.720000,4.31,3.908571,4.17,2.731429
1,11-1011.03,Chief Sustainability Officers,4.22,3.748571,2.74,2.525714,3.93,3.731429,4.48,4.491429,...,3.26,3.434286,4.33,3.771429,3.63,3.411429,4.30,4.194286,4.15,2.840000
2,11-1021.00,General and Operations Managers,4.04,3.440000,3.02,2.731429,3.91,3.588571,3.57,3.525714,...,3.70,3.474286,3.70,3.257143,3.41,3.165714,3.80,3.382857,4.46,3.371429
3,11-2011.00,Advertising and Promotions Managers,3.26,3.320000,2.37,2.142857,2.62,2.874286,4.37,4.257143,...,2.29,2.320000,4.05,3.702857,2.74,2.617143,3.78,3.565714,4.61,3.251429
4,11-2021.00,Marketing Managers,3.75,3.308571,2.44,2.262857,3.36,3.137143,4.20,4.200000,...,2.25,2.245714,4.28,3.691429,3.14,2.908571,3.83,3.582857,4.40,2.782857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,3.68,3.531429,2.54,2.228571,2.98,2.577143,3.04,2.531429,...,1.84,1.400000,3.12,3.011429,3.73,3.411429,4.05,3.714286,3.85,2.982857
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",3.96,3.268571,3.09,2.474286,2.86,2.537143,2.74,2.714286,...,2.18,2.177143,3.41,3.280000,3.15,2.737143,3.99,3.491429,2.69,2.131429
891,53-7073.00,Wellhead Pumpers,3.70,3.171429,3.04,2.697143,3.16,2.891429,2.72,2.394286,...,1.94,1.948571,3.34,3.011429,3.48,3.034286,3.31,3.228571,3.53,3.028571
892,53-7081.00,Refuse and Recyclable Material Collectors,1.61,1.588571,1.69,1.628571,2.10,2.028571,2.39,2.394286,...,1.00,1.005714,1.57,1.560000,1.69,1.697143,2.17,2.365714,1.55,1.548571


In [10]:
df_knowledge_scaled

,O*NET-SOC Code,Title,Analyzing Data or Information Importance,Analyzing Data or Information Level,Assisting and Caring for Others Importance,Assisting and Caring for Others Level,Coaching and Developing Others Importance,Coaching and Developing Others Level,Communicating with People Outside the Organization Importance,Communicating with People Outside the Organization Level,...,Staffing Organizational Units Importance,Staffing Organizational Units Level,Thinking Creatively Importance,Thinking Creatively Level,Training and Teaching Others Importance,Training and Teaching Others Level,Updating and Using Relevant Knowledge Importance,Updating and Using Relevant Knowledge Level,Working with Computers Importance,Working with Computers Level
0,11-1011.00,Chief Executives,4.38,3.765714,3.31,2.771429,4.71,4.342857,4.55,4.125714,...,3.78,3.742857,4.32,3.542857,4.01,3.720000,4.31,3.908571,4.17,2.731429
1,11-1011.03,Chief Sustainability Officers,4.22,3.748571,2.74,2.525714,3.93,3.731429,4.48,4.491429,...,3.26,3.434286,4.33,3.771429,3.63,3.411429,4.30,4.194286,4.15,2.840000
2,11-1021.00,General and Operations Managers,4.04,3.440000,3.02,2.731429,3.91,3.588571,3.57,3.525714,...,3.70,3.474286,3.70,3.257143,3.41,3.165714,3.80,3.382857,4.46,3.371429
3,11-2011.00,Advertising and Promotions Managers,3.26,3.320000,2.37,2.142857,2.62,2.874286,4.37,4.257143,...,2.29,2.320000,4.05,3.702857,2.74,2.617143,3.78,3.565714,4.61,3.251429
4,11-2021.00,Marketing Managers,3.75,3.308571,2.44,2.262857,3.36,3.137143,4.20,4.200000,...,2.25,2.245714,4.28,3.691429,3.14,2.908571,3.83,3.582857,4.40,2.782857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,3.68,3.531429,2.54,2.228571,2.98,2.577143,3.04,2.531429,...,1.84,1.400000,3.12,3.011429,3.73,3.411429,4.05,3.714286,3.85,2.982857
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",3.96,3.268571,3.09,2.474286,2.86,2.537143,2.74,2.714286,...,2.18,2.177143,3.41,3.280000,3.15,2.737143,3.99,3.491429,2.69,2.131429
891,53-7073.00,Wellhead Pumpers,3.70,3.171429,3.04,2.697143,3.16,2.891429,2.72,2.394286,...,1.94,1.948571,3.34,3.011429,3.48,3.034286,3.31,3.228571,3.53,3.028571
892,53-7081.00,Refuse and Recyclable Material Collectors,1.61,1.588571,1.69,1.628571,2.10,2.028571,2.39,2.394286,...,1.00,1.005714,1.57,1.560000,1.69,1.697143,2.17,2.365714,1.55,1.548571


In [11]:
df_work_context_scaled

,O*NET-SOC Code,Title,Conflict Situations,Consequence of Error,Contact With Others,Coordinate or Lead Others in Accomplishing Work Activities,Deal With External Customers or the Public in General,"Dealing With Unpleasant, Angry, or Discourteous People",Dealing with Violent or Physically Aggressive People,Degree of Automation,...,Spend Time Standing,"Spend Time Using Your Hands to Handle, Control, or Feel Objects, Tools, or Controls",Spend Time Walking or Running,Telephone Conversations,Time Pressure,"Wear Common Protective or Safety Equipment such as Safety Shoes, Glasses, Gloves, Hearing Protection, Hard Hats, or Life Jackets","Wear Specialized Protective or Safety Equipment such as Breathing Apparatus, Safety Harness, Full Protection Suits, or Radiation Protection",Work Outcomes and Results of Other Workers,Work With or Contribute to a Work Group or Team,Written Letters and Memos
0,11-1011.00,Chief Executives,3.86,3.02,4.58,4.29,4.33,3.24,1.46,2.23,...,2.13,2.17,1.72,4.92,4.16,1.77,1.21,4.70,4.36,4.14
1,11-1011.03,Chief Sustainability Officers,3.04,2.15,4.41,4.22,3.85,2.41,1.19,1.56,...,2.31,1.73,1.89,4.74,3.48,1.89,1.37,3.85,4.78,3.44
2,11-1021.00,General and Operations Managers,3.35,2.52,4.63,4.48,4.49,3.24,1.62,2.31,...,2.90,2.42,2.53,4.76,4.08,3.06,1.18,4.33,4.62,4.01
3,11-2011.00,Advertising and Promotions Managers,3.15,2.19,4.68,4.11,4.19,2.92,1.14,2.01,...,2.11,2.07,1.97,4.80,4.40,1.24,1.00,3.80,4.47,3.78
4,11-2021.00,Marketing Managers,3.18,2.76,4.61,4.32,4.25,2.74,1.16,2.10,...,2.09,2.31,1.95,4.92,4.21,1.30,1.04,4.04,4.67,3.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.59,4.41,4.07,3.42,2.36,2.50,1.17,2.77,...,3.00,3.65,3.10,4.76,3.76,5.00,2.33,3.61,4.51,2.73
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.92,3.59,4.25,3.11,2.90,2.21,1.15,2.11,...,3.44,4.11,2.33,4.48,3.31,4.99,1.55,3.03,3.85,2.38
891,53-7073.00,Wellhead Pumpers,2.62,3.67,4.12,3.13,3.43,2.52,1.38,2.00,...,2.95,3.38,2.59,4.95,3.52,5.00,1.59,3.24,3.81,3.20
892,53-7081.00,Refuse and Recyclable Material Collectors,2.44,3.17,3.61,2.56,2.78,2.91,1.34,2.33,...,3.09,4.15,2.66,2.80,3.74,4.99,1.42,2.21,2.93,1.62


# Combine All

In [12]:
# df_skills_scaled
# df_abilities_scaled
# df_activities_scaled
# df_knowledge_scaled
# df_work_context_scaled

from functools import reduce


dfs = [df_skills_scaled, df_abilities_scaled, df_activities_scaled, df_knowledge_scaled, df_work_context_scaled]

merge_cols = ['O*NET-SOC Code', 'Title']

merged_df = reduce(lambda left, right: pd.merge(left, right, on=merge_cols, how='inner'), dfs)
merged_df

,O*NET-SOC Code,Title,Active Learning Importance,Active Learning Level,Active Listening Importance,Active Listening Level,Complex Problem Solving Importance,Complex Problem Solving Level,Coordination Importance,Coordination Level,...,Spend Time Standing,"Spend Time Using Your Hands to Handle, Control, or Feel Objects, Tools, or Controls",Spend Time Walking or Running,Telephone Conversations,Time Pressure,"Wear Common Protective or Safety Equipment such as Safety Shoes, Glasses, Gloves, Hearing Protection, Hard Hats, or Life Jackets","Wear Specialized Protective or Safety Equipment such as Breathing Apparatus, Safety Harness, Full Protection Suits, or Radiation Protection",Work Outcomes and Results of Other Workers,Work With or Contribute to a Work Group or Team,Written Letters and Memos
0,11-1011.00,Chief Executives,3.75,3.571429,4.00,3.714286,4.38,3.788571,4.25,3.788571,...,2.13,2.17,1.72,4.92,4.16,1.77,1.21,4.70,4.36,4.14
1,11-1011.03,Chief Sustainability Officers,3.75,3.217143,4.00,3.285714,4.00,3.354286,3.75,3.217143,...,2.31,1.73,1.89,4.74,3.48,1.89,1.37,3.85,4.78,3.44
2,11-1021.00,General and Operations Managers,3.62,3.142857,4.00,3.354286,3.62,3.217143,3.88,3.285714,...,2.90,2.42,2.53,4.76,4.08,3.06,1.18,4.33,4.62,4.01
3,11-2011.00,Advertising and Promotions Managers,3.25,3.354286,4.12,3.354286,3.50,3.217143,3.50,3.354286,...,2.11,2.07,1.97,4.80,4.40,1.24,1.00,3.80,4.47,3.78
4,11-2021.00,Marketing Managers,3.88,3.354286,3.88,3.354286,3.62,3.217143,3.50,3.142857,...,2.09,2.31,1.95,4.92,4.21,1.30,1.04,4.04,4.67,3.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.88,2.360000,3.12,2.714286,3.00,2.714286,3.00,2.645714,...,3.00,3.65,3.10,4.76,3.76,5.00,2.33,3.61,4.51,2.73
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",2.88,2.645714,3.12,2.714286,2.88,2.645714,2.88,2.714286,...,3.44,4.11,2.33,4.48,3.31,4.99,1.55,3.03,3.85,2.38
891,53-7073.00,Wellhead Pumpers,2.38,2.211429,2.88,2.571429,3.00,2.645714,2.25,2.360000,...,2.95,3.38,2.59,4.95,3.52,5.00,1.59,3.24,3.81,3.20
892,53-7081.00,Refuse and Recyclable Material Collectors,2.25,1.925714,2.88,2.428571,2.38,2.211429,2.62,2.428571,...,3.09,4.15,2.66,2.80,3.74,4.99,1.42,2.21,2.93,1.62


In [13]:
merged_df.to_csv('transformed data/2-Prescaled Elements normalized.csv', index=False)

In [14]:
merged_df.columns

Index(['O*NET-SOC Code', 'Title', 'Active Learning Importance',
       'Active Learning Level', 'Active Listening Importance',
       'Active Listening Level', 'Complex Problem Solving Importance',
       'Complex Problem Solving Level', 'Coordination Importance',
       'Coordination Level',
       ...
       'Spend Time Standing',
       'Spend Time Using Your Hands to Handle, Control, or Feel Objects, Tools, or Controls',
       'Spend Time Walking or Running', 'Telephone Conversations',
       'Time Pressure',
       'Wear Common Protective or Safety Equipment such as Safety Shoes, Glasses, Gloves, Hearing Protection, Hard Hats, or Life Jackets',
       'Wear Specialized Protective or Safety Equipment such as Breathing Apparatus, Safety Harness, Full Protection Suits, or Radiation Protection',
       'Work Outcomes and Results of Other Workers',
       'Work With or Contribute to a Work Group or Team',
       'Written Letters and Memos'],
      dtype='object', length=395)

In [17]:
merged_df

,O*NET-SOC Code,Title,Active Learning Importance,Active Learning Level,Active Listening Importance,Active Listening Level,Complex Problem Solving Importance,Complex Problem Solving Level,Coordination Importance,Coordination Level,...,Spend Time Standing,"Spend Time Using Your Hands to Handle, Control, or Feel Objects, Tools, or Controls",Spend Time Walking or Running,Telephone Conversations,Time Pressure,"Wear Common Protective or Safety Equipment such as Safety Shoes, Glasses, Gloves, Hearing Protection, Hard Hats, or Life Jackets","Wear Specialized Protective or Safety Equipment such as Breathing Apparatus, Safety Harness, Full Protection Suits, or Radiation Protection",Work Outcomes and Results of Other Workers,Work With or Contribute to a Work Group or Team,Written Letters and Memos
0,11-1011.00,Chief Executives,3.75,3.571429,4.00,3.714286,4.38,3.788571,4.25,3.788571,...,2.13,2.17,1.72,4.92,4.16,1.77,1.21,4.70,4.36,4.14
1,11-1011.03,Chief Sustainability Officers,3.75,3.217143,4.00,3.285714,4.00,3.354286,3.75,3.217143,...,2.31,1.73,1.89,4.74,3.48,1.89,1.37,3.85,4.78,3.44
2,11-1021.00,General and Operations Managers,3.62,3.142857,4.00,3.354286,3.62,3.217143,3.88,3.285714,...,2.90,2.42,2.53,4.76,4.08,3.06,1.18,4.33,4.62,4.01
3,11-2011.00,Advertising and Promotions Managers,3.25,3.354286,4.12,3.354286,3.50,3.217143,3.50,3.354286,...,2.11,2.07,1.97,4.80,4.40,1.24,1.00,3.80,4.47,3.78
4,11-2021.00,Marketing Managers,3.88,3.354286,3.88,3.354286,3.62,3.217143,3.50,3.142857,...,2.09,2.31,1.95,4.92,4.21,1.30,1.04,4.04,4.67,3.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.88,2.360000,3.12,2.714286,3.00,2.714286,3.00,2.645714,...,3.00,3.65,3.10,4.76,3.76,5.00,2.33,3.61,4.51,2.73
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",2.88,2.645714,3.12,2.714286,2.88,2.645714,2.88,2.714286,...,3.44,4.11,2.33,4.48,3.31,4.99,1.55,3.03,3.85,2.38
891,53-7073.00,Wellhead Pumpers,2.38,2.211429,2.88,2.571429,3.00,2.645714,2.25,2.360000,...,2.95,3.38,2.59,4.95,3.52,5.00,1.59,3.24,3.81,3.20
892,53-7081.00,Refuse and Recyclable Material Collectors,2.25,1.925714,2.88,2.428571,2.38,2.211429,2.62,2.428571,...,3.09,4.15,2.66,2.80,3.74,4.99,1.42,2.21,2.93,1.62


# Sorting into the seven attributes

In [34]:
import re
import pandas as pd

def strip_suffixes(df):
    df2 = df.copy()
    df2.columns = [re.sub(r'_(x|y)$', '', c) for c in df2.columns]
    return df2

def dedupe_same_named_columns(df, method="first_nonnull"):
    """
    If duplicate column names exist, collapse them into a single column.
    method:
      - "first_nonnull": take first non-null across duplicates per row (safe default)
      - "mean": average duplicates per row (only if numeric)
    """
    if not df.columns.duplicated().any():
        return df.copy()

    out = pd.DataFrame(index=df.index)
    for name in pd.unique(df.columns):
        block = df.loc[:, df.columns == name]  # DataFrame of all dupes

        if block.shape[1] == 1:
            out[name] = block.iloc[:, 0]
        else:
            if method == "mean":
                out[name] = block.mean(axis=1, skipna=True)
            else:  # first_nonnull
                out[name] = block.bfill(axis=1).iloc[:, 0]
    return out

def combine_level_importance_1to5(df, drop_original=True):
    df_new = df.copy()

    level_cols = [c for c in df_new.columns if c.endswith(" Level")]
    imp_cols   = [c for c in df_new.columns if c.endswith(" Importance")]

    level_bases = {c.replace(" Level", ""): c for c in level_cols}
    imp_bases   = {c.replace(" Importance", ""): c for c in imp_cols}

    paired = set(level_bases).intersection(imp_bases)

    for element in paired:
        lcol = level_bases[element]
        icol = imp_bases[element]

        # 1–5 -> 0–1
        l_norm = (df_new[lcol] - 1) / 4
        i_norm = (df_new[icol] - 1) / 4

        # multiply, then back to 1–5
        df_new[f"{element} Combined"] = 1 + 4 * (l_norm * i_norm)

        if drop_original:
            df_new.drop(columns=[lcol, icol], inplace=True)

    return df_new


# ---- RUN ----
df_clean = strip_suffixes(merged_df)
df_clean = dedupe_same_named_columns(df_clean, method="first_nonnull")
df_combined = combine_level_importance_1to5(df_clean, drop_original=True)

/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_32221/3069648778.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = block.iloc[:, 0]
/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_32221/3069648778.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = block.iloc[:, 0]
/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_32221/3069648778.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor pe

In [38]:
df_combined.to_csv('transformed data/2-Prescaled Elements normalized comb level and import.csv', index=False)

In [36]:
# Old thing to read filenames. Likely won't be needed again.
import os

def print_filenames(folder_path):
    """
    Prints all filenames in the given folder (non-recursive).
    Only prints files, not subfolders.
    """
    try:
        for item in os.listdir(folder_path):
            full_path = os.path.join(folder_path, item)
            if os.path.isfile(full_path):
                print(item)
    except FileNotFoundError:
        print("Folder not found.")
    except Exception as e:
        print(f"Error: {e}")

print_filenames("ONET data/")

Interests Illustrative Occupations.csv
Work Activities.csv
Task Statements.csv
Skills to Work Activities.csv
Interests Illustrative Activities.csv
Task Categories.csv
Scales Reference.csv
Occupation Level Metadata.csv
Alternate Titles.csv
Tools Used.csv
Level Scale Anchors.csv
Technology Skills.csv
Content Model Reference.csv
Task Ratings.csv
Skills.csv
IWA Reference.csv
UNSPSC Reference.csv
Emerging Tasks.csv
Job Zones.csv
RIASEC Keywords.csv
Survey Booklet Locations.csv
Related Occupations.csv
Interests.csv
Job Zone Reference.csv
Abilities.csv
Education, Training, and Experience.csv
Work Values.csv
Basic Interests to RIASEC.csv
Abilities to Work Activities.csv
Tasks to DWAs.csv
Work Styles.csv
Sample of Reported Titles.csv
Work Context Categories.csv
Skills to Work Context.csv
Work Context.csv
DWA Reference.csv
Abilities to Work Context.csv
Education, Training, and Experience Categories.csv
Knowledge.csv
Occupation Data.csv
